<a href="https://colab.research.google.com/github/fidlarsyn/Scikit-learn-Cookbook--O-Reilly-/blob/main/12_Cross_Validation_and_Model_Evaluation_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction to cross-validation**
Validasi silang (cross-validation) adalah teknik statistik untuk mengevaluasi kemampuan generalisasi model dengan lebih stabil dibandingkan satu kali pembagian train-test split.
* **Prinsip Kerja**: Dataset dibagi menjadi k bagian (disebut folds). Model dilatih pada k−1 bagian dan diuji pada satu bagian sisanya. Proses ini diulang k kali sehingga setiap data point pernah menjadi bagian dari set pengujian.
* **Keuntungan**: Memberikan estimasi performa yang lebih andal, mengurangi risiko "beruntung" atau "sial" saat pembagian data acak, dan menggunakan data secara lebih efektif

# **Advanced cross-validation methods**
scikit-learn menyediakan berbagai strategi pembagian data untuk menangani karakteristik dataset yang berbeda:
* **Stratified k-Fold**: Memastikan proporsi kelas pada setiap fold sama dengan proporsi pada dataset utuh. Sangat disarankan untuk klasifikasi, terutama pada data yang tidak seimbang (imbalanced).
* **Leave-One-Out (LOOCV)**: Setiap fold adalah satu sampel tunggal. Metode ini sangat baik untuk dataset kecil karena memaksimalkan jumlah data pelatihan, namun sangat lambat pada dataset besar.
* **GroupKFold**: Digunakan jika data mengandung kelompok yang sangat terkait (misal: beberapa foto dari orang yang sama). Metode ini memastikan anggota kelompok yang sama tidak terpisah antara set pelatihan dan pengujian untuk benar-benar menguji generalisasi pada kelompok baru.
* **Nested Cross-Validation**: Menggunakan loop dalam untuk pencarian parameter (grid search) dan loop luar untuk evaluasi. Ini mencegah kebocoran informasi dari proses penyetelan parameter ke hasil evaluasi akhir.

# **Model evaluation metrics overview**
Memilih metrik yang tepat sangat bergantung pada tujuan bisnis.

**Klasifikasi**:
* **Akurasi**: Rasio prediksi benar terhadap total sampel. Bisa menyesatkan pada data imbalanced.
* **Presisi**: Ketepatan model dalam mengidentifikasi kelas positif.
* **Recall (Sensitivitas)**: Kemampuan model menemukan seluruh instans positif yang ada.
* **F1-score**: Rata-rata harmonik antara presisi dan recall.
* **ROC Curve & AUC**: Mengukur kemampuan model membedakan antar kelas pada berbagai ambang batas keputusan.

**Regresi**:
* **R² (Koefisien Determinasi)**: Proporsi varians target yang dapat dijelaskan oleh fitur.
* **Mean Squared Error (MSE)**: Rata-rata kuadrat kesalahan prediksi.


# **Implementing cross-validation in scikit-learn**
Fungsi praktis untuk menghitung skor validasi secara otomatis.


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# 1. Validasi Silang Sederhana
model = LogisticRegression()
scores = cross_val_score(model, X, y, cv=5) # 5-fold CV
print(f"Rata-rata Akurasi: {np.mean(scores):.3f}")

# 2. Menggunakan Multiple Metrics
results = cross_validate(model, X, y, cv=5, scoring=['precision', 'recall', 'f1'])
print(f"Mean F1 Score: {np.mean(results['test_f1']):.3f}")

# 3. Menggunakan StratifiedKFold secara eksplisit
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2024)
strat_scores = cross_val_score(model, X, y, cv=skf)

# **Model selection techniques**
Proses pemilihan model melibatkan pengujian berbagai kombinasi hyperparameter untuk menemukan yang terbaik.
* **Grid Search (GridSearchCV)**: Menguji secara menyeluruh setiap kombinasi parameter dalam "kisi" yang ditentukan.
* **Randomized Search (RandomizedSearchCV)**: Mengambil sampel acak dari distribusi parameter. Lebih efisien dan seringkali memberikan hasil yang sebanding dengan waktu yang jauh lebih cepat.

**Contoh Kode:**

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]}
grid = GridSearchCV(SVC(), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Parameter Terbaik:", grid.best_params_)
print("Skor Terbaik:", grid.best_score_)

# **Evaluating model generalizability**
Generalisasi adalah kemampuan model untuk berkinerja baik pada data yang belum pernah dilihat sebelumnya.
* **Kurva Pembelajaran (Learning Curves)**: Menunjukkan bagaimana performa berubah seiring bertambahnya ukuran data. Membantu mendeteksi apakah model membutuhkan lebih banyak data atau sudah mencapai titik jenuh.
* **Kurva Validasi (Validation Curves)**: Menunjukkan pengaruh satu hyperparameter terhadap skor pelatihan dan validasi untuk mendeteksi overfitting atau underfitting.
* **Pentingnya Pipeline**: Selalu gunakan Pipeline saat melakukan validasi silang jika ada proses prapemrosesan (seperti scaling). Ini memastikan prapemrosesan hanya dipelajari dari set pelatihan di setiap fold, menghindari kebocoran data (data leakage).

**Contoh Kode:**

In [ ]:
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt

train_sizes, train_scores, test_scores = learning_curve(
    LogisticRegression(), X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 5)
)

plt.plot(train_sizes, np.mean(train_scores, axis=1), label='Training score')
plt.plot(train_sizes, np.mean(test_scores, axis=1), label='Cross-validation score')
plt.xlabel('Training set size')
plt.ylabel('Accuracy')
plt.show()